# Ingest RAND Longitudinal SPSS Files to Databricks

**Source:** `D:\OneDrive\Documents\Databricks Training\HRS\RAND Longitudinal\randhrs1992_2022v1_SPSS`  

**Destination Volume:** `landing_catalog.hrs_vol`  
---

## Important Note
Databricks notebooks run on cloud compute and **cannot directly access your local PC files**. You must first upload files from your local directory to Databricks.


### UI Upload (Easiest and the method I used)

1. Download the randhrs1992_2020v1_spss file from the RAND HRS website
2. Open **Catalog Explorer** and create → `landing_catalog` → `external_data`
3. Click **Upload** button
4. Select `.sav` file(s) from your local directory
5. Wait for upload to complete
---

# Design

```text
SPSS (.sav)
      │
      ▼
Read in Chunks (5000 rows)
      │
      ▼
Write each chunk as a Parquet file
      │
      ├── chunk_00001.parquet
      ├── chunk_00002.parquet
      ├── chunk_00003.parquet
      └── ...
      │
      ▼
Checkpoint file records last successful chunk
      │
      ▼
If cluster dies...
      │
      ▼
Restart notebook
      │
      ▼
Resume at next chunk
      │
      ▼
When complete:
Read all Parquet files
      │
      ▼
Write Delta Table

In [ ]:
%pip install pyreadstat
%restart_python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip
UsageError: Line magic function `%restart_python` not found.


In [0]:
# List source files to verify the upload
import os

volume_path = "/Volumes/landing_catalog/external_data/rand_hrs_raw_data/randhrs1992_2022v1.sav"

print("Files in volume:")
print("=" * 80)

files = dbutils.fs.ls(volume_path)
for file_info in files:
    size_mb = file_info.size / (1024 * 1024)
    print(f"{file_info.name:50s} {size_mb:>10.2f} MB")

print(f"\nTotal files: {len(files)}")

In [ ]:
# SPSS to Parquet (Restartable)  
# ==============================================================================
# HRS Bronze Ingestion
#
# Reads a large SPSS file in chunks.
# Each chunk is written as an individual Parquet dataset.
# Progress is checkpointed so the notebook can resume after a cluster restart.
#
# AI Agent Assisted: ChatGPT
# ==============================================================================

import os
import pyreadstat

# ------------------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------------------

sav_file = "/Volumes/landing_catalog/external_data/rand_hrs_raw_data/randhrs1992_2022v1.sav"

parquet_folder = "/Volumes/landing_catalog/external_data/rand_hrs_raw_data/parquet_chunks"

checkpoint_file = f"{parquet_folder}/_checkpoint.txt"

chunk_size = 5000

# ------------------------------------------------------------------------------
# Create destination folder
# ------------------------------------------------------------------------------

dbutils.fs.mkdirs(parquet_folder)

# ------------------------------------------------------------------------------
# Determine starting chunk
# ------------------------------------------------------------------------------

start_chunk = 1

try:
    checkpoint = dbutils.fs.head(checkpoint_file)

    start_chunk = int(checkpoint.strip()) + 1

    print(f"Resuming from chunk {start_chunk}")

except:

    print("No checkpoint found.")
    print("Starting from beginning.")

# ------------------------------------------------------------------------------
# Read SPSS
# ------------------------------------------------------------------------------

reader = pyreadstat.read_file_in_chunks(
    pyreadstat.read_sav,
    sav_file,
    chunksize=chunk_size
)

# ------------------------------------------------------------------------------
# Process chunks
# ------------------------------------------------------------------------------

for chunk_number, (pdf, meta) in enumerate(reader, start=1):

    if chunk_number < start_chunk:
        continue

    print(f"Processing chunk {chunk_number}")

    sdf = spark.createDataFrame(pdf)

    chunk_path = f"{parquet_folder}/chunk_{chunk_number:05d}"

    sdf.write \
        .mode("overwrite") \
        .parquet(chunk_path)

    # Save checkpoint AFTER successful write
    dbutils.fs.put(
        checkpoint_file,
        str(chunk_number),
        overwrite=True
    )

    print(f"Completed chunk {chunk_number}")

print("All chunks written successfully.")

In [ ]:
# Parquet to Delta Table
#==============================================================================
# HRS Bronze Ingestion
#
# Reads parquet chunks.
# Build the Delta Table
# 
# ==============================================================================

table_name = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

parquet_folder = "/Volumes/landing_catalog/external_data/rand_hrs_raw_data/parquet_chunks"

spark.sql(f"DROP TABLE IF EXISTS {table_name}")

(
    spark.read
         .parquet(parquet_folder + "/chunk_*")
         .write
         .format("delta")
         .mode("overwrite")
         .saveAsTable(table_name)
)

print("Delta table created.")

record_count = spark.table(table_name).count()

print(f"Rows Loaded      : {record_count:,}")

In [ ]:
# Confirm Table Creation
#==============================================================================
# Dispaly 10 rows
#==============================================================================

df = spark.table("dev_catalog.brz_raw_hrs.randhrs1992_2022v1")

display(df.limit(10))

In [ ]:
# Parquet Clean Up
#==============================================================================
# Remove Parquet Chuncks
# =============================================================================

dbutils.fs.rm(
    "/Volumes/landing_catalog/external_data/rand_hrs_raw_data/parquet_chunks",
    recurse=True
)

print("Temporary Parquet chunks removed.")